# F4 — Análisis de resultados SIMCE 2025
**4° Básico · Matemática**

Este cuaderno implementa el análisis final acordado para responder las tres preguntas de investigación:

1. ¿Existen diferencias estadísticamente significativas en los puntajes promedio según ubicación geográfica?
2. ¿Qué regiones, provincias y comunas concentran establecimientos con desempeño inferior a la referencia nacional?
3. ¿Qué establecimientos y territorios pueden identificarse como potenciales focos de priorización?

### Decisiones metodológicas
- La **unidad de análisis inferencial** es el establecimiento.
- Para comparar establecimientos territorialmente se usa el **promedio simple**.
- Para estimar desempeño territorial y construir la referencia de criticidad se usa el **promedio ponderado por estudiantes**.
- El ICT comunal combina **déficit (50%) + concentración (30%) + exposición (20%)**, normalizados mediante percentiles.
- Se selecciona el **P90 del ICT** como criterio de focalización relativa.
- La sensibilidad se comprueba excluyendo comunas bajo P10 del número de establecimientos.

## 1. Configuración, carga y validaciones

In [ ]:
from pathlib import Path
from importlib import metadata
import sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.oneway import anova_oneway, effectsize_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

candidatos = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidatos if (p/'src/poo.py').is_file() and (p/'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook dentro del repositorio SIMCE con src/ y data/.')
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.analisis_f4 import (validar_entrada, promedio_ponderado, resumen_criticidad,
    calcular_ict, prueba_territorial, REGION, COMUNA)

SEMILLA = 42
CUANTIL_CRITICO = 0.90
np.random.seed(SEMILLA)
OUT = ROOT/'F4/resultados'
OUT.mkdir(parents=True, exist_ok=True)
paquetes = ['numpy','pandas','matplotlib','nbformat','nbclient','ipykernel','ipython']
versiones = {p: metadata.version(p) for p in paquetes}
print('Python:', platform.python_version(), '| Plataforma:', platform.platform())
print('Entorno virtual:', sys.prefix != sys.base_prefix)
display(pd.DataFrame(versiones.items(), columns=['paquete','version']))
pd.set_option('display.max_columns', 14)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

## III. Continuidad y decisiones F2/F3

| Decisión mantenida | Alternativa descartada y razón específica |
|---|---|
| Efectiva: alumnos > 0 y sin observación de puntaje | Reincorporar marca 2 por tener puntaje contradice la advertencia de representatividad |
| No imputar RBD, geografía ni puntaje | Un valor inventado alteraría pertenencia territorial o prioridad de intervención |
| Conservar extremos válidos | Eliminar escuelas grandes o con puntaje extremo borraría precisamente casos relevantes |
| Mantener puntaje y alumnos en sus unidades | Un z-score no responde cuántos alumnos se asocian a cuántos puntos de brecha |
| Geografía embebida y unión validada | Una tabla externa adicional puede desalinear códigos entre ejecuciones |
| Errores detienen el proceso | Eliminar filas inválidas silenciosamente cambiaría el universo y los denominadores |

**Arquitectura y POO.** `Transformador` es una clase abstracta; cinco hijas implementan `transformar`. `PipelineSIMCE` compone esas instancias y las recorre por su interfaz común (polimorfismo). Las propiedades exponen copias o tuplas, evitando que el consumidor altere accidentalmente el registro interno. Se reutiliza esta arquitectura, sin crear otra jerarquía artificial en F4.

Separar `auditoria.py` de `validacion.py` permite conservar evidencia de exclusiones mientras los errores de integridad detienen la ejecución. `configuracion.py` centraliza catálogos para que las clases y las funciones compartan reglas. `poo.py` coordina las transformaciones sin duplicar su lógica. `analisis_f4.py` recibe el producto ya validado: cambiar un gráfico no debe modificar la regla de efectividad. Estas decisiones atienden las observaciones formativas del 25 de septiembre.

In [ ]:
from src.configuracion import COLUMNAS_SIMCE
from src.carga import cargar_simce, construir_dim_geografia
from src.poo import (PipelineSIMCE, TransformadorTipos, TransformadorTextos,
    TransformadorCategorias, TransformadorGeografia, TransformadorEfectividad)
from src.auditoria import construir_auditoria
from src.transformacion import construir_cobertura_regional
from src.validacion import validar_dataset_final
from src.algoritmo import construir_jerarquia_geografica, validar_geografia_recursiva

raw_path = ROOT/'data/raw/simce4b2025_rbd_final.csv'
f2_path = ROOT/'data/processed/simce4b2025_matematica_efectiva.csv'
df_f2 = pd.read_csv(f2_path, encoding='utf-8-sig', dtype={'no_aplica':'string'}, parse_dates=['fecha_bbdd'])
dim = construir_dim_geografia()
bruto = cargar_simce(raw_path, COLUMNAS_SIMCE)
pipeline = PipelineSIMCE([TransformadorTipos(), TransformadorTextos(),
    TransformadorCategorias(), TransformadorGeografia(dim), TransformadorEfectividad()])
transformado = pipeline.ejecutar(bruto)
reconstruido = pipeline.construir_producto_final(transformado)
auditoria, resumen_auditoria, resumen_marcas = construir_auditoria(transformado)
cobertura = construir_cobertura_regional(transformado)
controles_f3 = validar_dataset_final(reconstruido, transformado, auditoria, cobertura)
cols = sorted(reconstruido.columns)
pd.testing.assert_frame_equal(
    df_f2.sort_values('rbd').reset_index(drop=True)[cols],
    reconstruido.sort_values('rbd').reset_index(drop=True)[cols], check_dtype=False)
assert len(bruto) == len(df_f2) + len(auditoria)
jerarquia = construir_jerarquia_geografica(dim)
assert validar_geografia_recursiva(jerarquia) == []
df = validar_entrada(df_f2)
print(f'Bruto: {len(bruto):,}; efectivos: {len(df):,}; auditados: {len(auditoria):,}.')
print('Igualdad de los 30 campos F2/F3 comprobada; controles:', len(controles_f3))
display(pd.DataFrame(pipeline.registro))
display(cobertura)
comunas_ausentes = dim.loc[~dim.cod_com_rbd.isin(df.cod_com_rbd), ['cod_reg_rbd','region','cod_com_rbd','comuna']]
print('Comunas de la dimensión sin establecimientos efectivos:')
display(comunas_ausentes)